# **APPROACH 1 DEOLDIFY**

In [ ]:
#IMP
!pip install torch==2.1.1 torchvision==0.16.1 torchaudio==2.1.1 --force-reinstall

 Git clone and install DeOldify

In [ ]:
!git clone https://github.com/jantic/DeOldify.git DeOldify

fatal: destination path 'DeOldify' already exists and is not an empty directory.


In [ ]:
cd DeOldify

/content/DeOldify


# Setup


In [ ]:
#NOTE:  This must be the first call in order to work properly!
from deoldify import device
from deoldify.device_id import DeviceId
#choices:  CPU, GPU0...GPU7
device.set(device=DeviceId.GPU0)

import torch

if not torch.cuda.is_available():
    print('GPU not available.')

In [ ]:
#!pip install -r requirements-colab.txt

# # # Use specific versions for compatibility
!pip install fastai==1.0.61
!pip install torch==1.7.1 torchvision==0.8.2
!pip install ffmpeg-python
!pip install opencv-python

In [ ]:
import fastai
from deoldify.visualize import *
import warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*?Your .*? set is empty.*?")

INFO:numexpr.utils:NumExpr defaulting to 12 threads.


NumExpr defaulting to 12 threads.


In [ ]:
!mkdir 'models'
!wget --no-check-certificate https://data.deepai.org/deoldify/ColorizeArtistic_gen.pth -O ./models/ColorizeArtistic_gen.pth

In [ ]:
colorizer = get_image_colorizer(artistic=True)

# Colorize!!

In [ ]:
#This only need to be run once, then Restart Session, Then re run the code starting from this cell
# "!git clone https://github.com/jantic/DeOldify.git DeOldify "
# then DO NOT Re Run this cell..
# Start Colorizing the cell bellow..

!pip install numpy==1.26.4 --force-reinstall


In [ ]:
from google.colab import files
import zipfile, os
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# Upload ZIP
uploaded = files.upload()
zip_path = next(iter(uploaded))

# Extract
extract_dir = Path("input_images")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Output directory for colorized
output_dir = Path("colorized_images")
output_dir.mkdir(parents=True, exist_ok=True)

# Process each image
for img_name in os.listdir(extract_dir):
    input_path = extract_dir / img_name
    if input_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
        continue

    # Load and ensure RGB
    original_img = Image.open(input_path).convert('RGB')
    original_img.save(input_path)

    # Colorize
    colorized_img = colorizer.get_transformed_image(
        path=input_path, render_factor=35, watermarked=False
    )

    # Save colorized image
    colorized_path = output_dir / f"colorized_{img_name}"
    colorized_img.save(colorized_path)

    # Show side-by-side using matplotlib
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(original_img)
    axes[0].set_title("Pre-processing")
    axes[0].axis('off')

    axes[1].imshow(colorized_img)
    axes[1].set_title("Colorized")
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np
import cv2
from PIL import Image
import os
import matplotlib.pyplot as plt

# Ensure the output directory exists
def ensure_directory_exists(output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Created directory: {output_folder}")

# Apply Unsharp Masking for light sharpening
def apply_unsharp_mask(image, sigma=1.0, amount=1.5):
    # Convert image to float32 for processing
    image_float = image.astype(np.float32) / 255.0

    # Apply Gaussian blur to create a blurred image
    blurred = cv2.GaussianBlur(image_float, (0, 0), sigmaX=sigma, sigmaY=sigma)

    # Subtract the blurred image from the original (Unsharp Mask)
    sharpened = np.clip(image_float + amount * (image_float - blurred), 0, 1)

    # Convert back to 8-bit
    sharpened = (sharpened * 255).astype(np.uint8)

    return sharpened

# Post-process the image with sharpening
def sharpen_post_process_image(image):
    sharpened_image = apply_unsharp_mask(image)
    return sharpened_image

# Process and save post-processed images
def process_post_processing_and_save(image_path, output_folder):
    image = Image.open(image_path)
    image = image.convert('RGB')  # Ensure RGB mode
    image_np = np.array(image)

    # Apply sharpening post-processing
    processed_image = sharpen_post_process_image(image_np)

    # Convert back to PIL for saving
    processed_image_pil = Image.fromarray(processed_image)

    # Ensure the output directory exists
    ensure_directory_exists(output_folder)

    # Save the processed image
    output_image_path = os.path.join(output_folder, f"processed_{os.path.basename(image_path)}")
    processed_image_pil.save(output_image_path)

    print(f"Processed and saved post-processed image: {output_image_path}")

def process_all_colorized_images_for_post_processing(colorized_folder, output_folder):
    image_files = os.listdir(colorized_folder)

    for image_file in image_files:
        image_path = os.path.join(colorized_folder, image_file)
        process_post_processing_and_save(image_path, output_folder)

colorized_base_path = '/content/drive/My Drive/DeOldified_Results/'  # Colorized images folder
output_folder = '/content/drive/My Drive/PostProcessed2_Images/'

process_all_colorized_images_for_post_processing(colorized_base_path, output_folder)

# Visualization code to display images side by side (colorized vs post-processed)
def visualize_images(colorized_image_path, post_processed_image_path):
    # Open colorized and post-processed images
    colorized_image = Image.open(colorized_image_path)
    colorized_image = np.array(colorized_image)

    post_processed_image = Image.open(post_processed_image_path)
    post_processed_image = np.array(post_processed_image)

    # Plot the images side by side
    plt.figure(figsize=(10, 5))

    # Colorized Image
    plt.subplot(1, 2, 1)
    plt.imshow(colorized_image)
    plt.title('Colorized Image')
    plt.axis('off')

    # Post-Processed Image
    plt.subplot(1, 2, 2)
    plt.imshow(post_processed_image)
    plt.title('Post-Processed Image')
    plt.axis('off')

    plt.show()


In [ ]:
import shutil

shutil.make_archive("Colorized_Results", 'zip', output_dir)
files.download("Colorized_Results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install lpips

In [ ]:
import os
from pathlib import Path
from PIL import Image
import numpy as np
from skimage.metrics import structural_similarity as ssim
import math
import lpips
import torch

# Correct paths based on your actual folders
original_dir = Path('input_images')        # <-- You had this as original_images (wrong)
colorized_dir = Path('colorized_images')

# Initialize LPIPS model
lpips_model = lpips.LPIPS(net='alex')  # Choose from 'alex', 'vgg', or 'squeeze'

def compute_psnr(img1, img2):
    mse = np.mean((np.array(img1) - np.array(img2)) ** 2)
    if mse == 0:
        return float('inf')
    PIXEL_MAX = 255.0
    return 20 * math.log10(PIXEL_MAX / math.sqrt(mse))

# Metric lists
psnr_values, ssim_values, lpips_values = [], [], []

# Evaluate all image pairs
for img_name in os.listdir(original_dir):
    orig_path = original_dir / img_name
    color_path = colorized_dir / f"colorized_{img_name}"  # Match colorized filename pattern

    if not color_path.exists():
        print(f"Skipping {img_name} (no colorized match)")
        continue

    # Load images and ensure RGB
    orig_img = Image.open(orig_path).convert('RGB')
    color_img = Image.open(color_path).convert('RGB')

    # Resize if needed
    if orig_img.size != color_img.size:
        color_img = color_img.resize(orig_img.size)

    # Convert to NumPy
    orig_np = np.array(orig_img)
    color_np = np.array(color_img)

    # Compute metrics
    psnr_val = compute_psnr(orig_np, color_np)
    ssim_val = ssim(orig_np, color_np, channel_axis=-1)

    orig_tensor = torch.tensor(orig_np).permute(2, 0, 1).unsqueeze(0).float() / 255.0
    color_tensor = torch.tensor(color_np).permute(2, 0, 1).unsqueeze(0).float() / 255.0
    lpips_val = lpips_model(orig_tensor, color_tensor).item()

    # Append values
    psnr_values.append(psnr_val)
    ssim_values.append(ssim_val)
    lpips_values.append(lpips_val)

# Report
print(f"\nAverage PSNR:  {np.mean(psnr_values):.2f} dB")
print(f"Average SSIM:  {np.mean(ssim_values):.4f}")
print(f"Average LPIPS: {np.mean(lpips_values):.4f}")


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/alex.pth
Skipping High (no colorized match)
Skipping Medium (no colorized match)
Skipping Low (no colorized match)
Skipping preprocessed_images_ShadenHigh (no colorized match)

Average PSNR:  30.98 dB
Average SSIM:  0.9838
Average LPIPS: 0.0918


# Post Processing

In [ ]:
import numpy as np
import cv2
from PIL import Image
import os
import matplotlib.pyplot as plt

# Ensure the output directory exists
def ensure_directory_exists(output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Created directory: {output_folder}")

# Apply Unsharp Masking for light sharpening
def apply_unsharp_mask(image, sigma=1.0, amount=1.5):
    # Convert image to float32 for processing
    image_float = image.astype(np.float32) / 255.0

    # Apply Gaussian blur to create a blurred image
    blurred = cv2.GaussianBlur(image_float, (0, 0), sigmaX=sigma, sigmaY=sigma)

    # Subtract the blurred image from the original (Unsharp Mask)
    sharpened = np.clip(image_float + amount * (image_float - blurred), 0, 1)

    # Convert back to 8-bit
    sharpened = (sharpened * 255).astype(np.uint8)

    return sharpened

# Post-process the image with sharpening
def sharpen_post_process_image(image):
    sharpened_image = apply_unsharp_mask(image)
    return sharpened_image

# Process and save post-processed images
def process_post_processing_and_save(image_path, output_folder):
    image = Image.open(image_path)
    image = image.convert('RGB')  # Ensure RGB mode
    image_np = np.array(image)

    # Apply sharpening post-processing
    processed_image = sharpen_post_process_image(image_np)

    # Convert back to PIL for saving
    processed_image_pil = Image.fromarray(processed_image)

    # Ensure the output directory exists
    ensure_directory_exists(output_folder)

    # Save the processed image
    output_image_path = os.path.join(output_folder, f"processed_{os.path.basename(image_path)}")
    processed_image_pil.save(output_image_path)

    print(f"Processed and saved post-processed image: {output_image_path}")

def process_all_colorized_images_for_post_processing(colorized_folder, output_folder):
    image_files = os.listdir(colorized_folder)

    for image_file in image_files:
        image_path = os.path.join(colorized_folder, image_file)
        process_post_processing_and_save(image_path, output_folder)

colorized_base_path = '/content/drive/My Drive/DeOldified_Results/'  # Colorized images folder
output_folder = '/content/drive/My Drive/PostProcessed2_Images/'

process_all_colorized_images_for_post_processing(colorized_base_path, output_folder)

# Visualization code to display images side by side (colorized vs post-processed)
def visualize_images(colorized_image_path, post_processed_image_path):
    # Open colorized and post-processed images
    colorized_image = Image.open(colorized_image_path)
    colorized_image = np.array(colorized_image)

    post_processed_image = Image.open(post_processed_image_path)
    post_processed_image = np.array(post_processed_image)

    # Plot the images side by side
    plt.figure(figsize=(10, 5))

    # Colorized Image
    plt.subplot(1, 2, 1)
    plt.imshow(colorized_image)
    plt.title('Colorized Image')
    plt.axis('off')

    # Post-Processed Image
    plt.subplot(1, 2, 2)
    plt.imshow(post_processed_image)
    plt.title('Post-Processed Image')
    plt.axis('off')

    plt.show()


In [ ]:
def visualize_all_colorized_and_post_processed_images(colorized_folder, post_processed_folder):
    colorized_files = os.listdir(colorized_folder)

    for file in colorized_files:
        colorized_image_path = os.path.join(colorized_folder, file)
        post_processed_image_path = os.path.join(post_processed_folder, f"processed_{file}")

        # Visualize the colorized and post-processed images side by side
        visualize_images(colorized_image_path, post_processed_image_path)

colorized_folder = '/content/drive/My Drive/DeOldified_Results/'  # Colorized images folder
post_processed_folder = '/content/drive/My Drive/PostProcessed2_Images/'  # Post-processed images folder

visualize_all_colorized_and_post_processed_images(colorized_folder, post_processed_folder)

# **APPROACH 2 DEOLDIFY**

In [ ]:
#Libraries
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Paths
base_path = '/content/drive/My Drive/Old_Images/'
high_quality_folder = os.path.join(base_path, 'High')
medium_quality_folder = os.path.join(base_path, 'Medium')
low_quality_folder = os.path.join(base_path, 'Low')

In [ ]:
# Display the original and the processed images side by side

def display_comparison(original, processed, title="Image Comparison"):
    plt.figure(figsize=(12, 6))

    #Original image
    plt.subplot(1, 2, 1)
    plt.imshow(original)
    plt.title("Original Image")
    plt.axis('off')

    #Processed image
    plt.subplot(1, 2, 2)
    plt.imshow(processed)
    plt.title("Processed Image")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
#CLAHE to color images
def apply_clahe_to_color(image):
    #Convert to LAB color space
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    #Apply CLAHE to the luminance channel
    clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)

    #Merge the channels and convert back to RGB
    limg = cv2.merge((cl, a, b))
    result = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

    return result

In [ ]:
#CLAHE to grayscale images
def apply_clahe_to_grayscale(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)  #Convert to grayscale
    clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(8, 8))
    enhanced_image = clahe.apply(gray)  #Apply CLAHE to the grayscale image
    return enhanced_image

In [ ]:
'''This is a method that preprocesses the
image based on its quality (which we have categorised)'''

'''cv2.fastNlMeansDenoising(src image, dst image - optional ,
h is the filter strength, templateWindowSize is the size of the square
patch that is used to compare each pixel's surrounding neighborhood,
searchWindowSize is the search area around each pixel to look for similar patches)'''

def preprocess_image(image, quality_category):
    if quality_category == 'High':
        #Minimal preprocessing
        image = cv2.fastNlMeansDenoising(image, None, h=5, templateWindowSize=7, searchWindowSize=21)

    elif quality_category == 'Medium':
        #Moderate preprocessing
        image = cv2.fastNlMeansDenoising(image, None, h=7, templateWindowSize=7, searchWindowSize=21)
        image = apply_clahe_to_color(image)  #Apply CLAHE to color image

    elif quality_category == 'Low':
        #Strong preprocessing
        image = cv2.fastNlMeansDenoising(image, None, h=6, templateWindowSize=7, searchWindowSize=21)
        #Light Sharpening
        kernel = np.array([[0, -0.2, 0], [-0.2, 2, -0.2], [0, -0.2, 0]])  # Milder sharpening kernel
        image = cv2.filter2D(image, -1, kernel)

        #Apply CLAHE only on the luminance channel (Y channel in YUV space) to avoid noise enhancement in color channels
        image = apply_clahe_to_color(image) #Apply CLAHE to color image

    return image

In [ ]:
#Method to process and display images

def process_and_display_images(folder_path, quality_category):
    #Get the list of images in the folder
    image_files = os.listdir(folder_path)

    #Process each image
    for image_file in image_files:
        image_path = os.path.join(folder_path, image_file)

        #Load the image
        image = cv2.imread(image_path)
        if image is None:
            print(f"Error: {image_file} not loaded properly.")
            continue

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  #Convert BGR to RGB for correct display

        #Apply preprocessing based on quality category
        processed_image = preprocess_image(image_rgb, quality_category)

        #Display the original and processed image side by side
        display_comparison(image_rgb, processed_image, title=f"{quality_category} Quality - {image_file}")


In [ ]:
process_and_display_images(high_quality_folder, 'High')
process_and_display_images(medium_quality_folder, 'Medium')
process_and_display_images(low_quality_folder, 'Low')

In [ ]:
# Function to preprocess and save images to a directory
def process_and_save_images(folder_path, quality_category, output_folder):
    # Get the list of images in the folder
    image_files = os.listdir(folder_path)

    # Ensure the output folder exists
    output_path = os.path.join(output_folder, quality_category)
    os.makedirs(output_path, exist_ok=True)

    # Process each image
    for image_file in image_files:
        image_path = os.path.join(folder_path, image_file)
        image = cv2.imread(image_path)

        if image is None:
            print(f"Error: {image_file} not loaded properly.")
            continue

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB

        # Apply preprocessing based on quality category
        processed_image = preprocess_image(image_rgb, quality_category)

        # Save the processed image
        output_image_path = os.path.join(output_path, image_file)
        cv2.imwrite(output_image_path, cv2.cvtColor(processed_image, cv2.COLOR_RGB2BGR))  # Convert back to BGR before saving

        print(f"Processed and saved: {image_file}")

In [ ]:
# Function to zip the processed images
def zip_images(output_folder, zip_name):

    #Create a zip file
    zip_path = f"{zip_name}.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(output_folder):
            for file in files:
                zipf.write(os.path.join(root, file), os.path.relpath(os.path.join(root, file), output_folder))

    print(f"Created zip file: {zip_path}")
    return zip_path

In [ ]:
import zipfile
drive.mount('/content/drive')


output_folder = '/content/drive/My Drive/Processed_Images' #Folder where processed images will be saved

#Process and save images
process_and_save_images(high_quality_folder, 'High', output_folder)
process_and_save_images(medium_quality_folder, 'Medium', output_folder)
process_and_save_images(low_quality_folder, 'Low', output_folder)

#Zip the processed images
zip_file = zip_images(output_folder, 'preprocessed_images')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/jantic/DeOldify.git DeOldify

fatal: destination path 'DeOldify' already exists and is not an empty directory.


In [ ]:
cd DeOldify

/content/DeOldify


In [ ]:
#NOTE:  This must be the first call in order to work properly!
from deoldify import device
from deoldify.device_id import DeviceId
#choices:  CPU, GPU0...GPU7
device.set(device=DeviceId.GPU0)

import torch

if not torch.cuda.is_available():
    print('GPU not available.')

GPU not available.


In [ ]:
#!pip install -r requirements-colab.txt

# Use specific versions for compatibility
!pip install fastai==1.0.61
!pip install torch==1.7.1 torchvision==0.8.2
!pip install ffmpeg-python
!pip install opencv-python

In [ ]:
!pip install yt-dlp

In [ ]:
import fastai
from deoldify.visualize import *
import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*?Your .*? set is empty.*?')

INFO:numexpr.utils:NumExpr defaulting to 2 threads.


NumExpr defaulting to 2 threads.


In [ ]:
!mkdir 'models'
!wget --no-check-certificate https://data.deepai.org/deoldify/ColorizeArtistic_gen.pth -O ./models/ColorizeArtistic_gen.pth

mkdir: cannot create directory ‘models’: File exists
--2025-05-02 21:01:34--  https://data.deepai.org/deoldify/ColorizeArtistic_gen.pth
Resolving data.deepai.org (data.deepai.org)... 185.93.1.242, 2400:52e0:1a00::940:1
Connecting to data.deepai.org (data.deepai.org)|185.93.1.242|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 255144681 (243M) [application/octet-stream]
Saving to: ‘./models/ColorizeArtistic_gen.pth’

./models/ColorizeAr 100%[===================>] 243.32M  4.88MB/s    in 50s     

2025-05-02 21:02:24 (4.89 MB/s) - ‘./models/ColorizeArtistic_gen.pth’ saved [255144681/255144681]



In [ ]:
colorizer = get_image_colorizer(artistic=True)

In [ ]:
#ONLY ONCE!!!!!!!
#IMP
!pip install torch==2.1.1 torchvision==0.16.1 torchaudio==2.1.1 --force-reinstall

In [ ]:
#ONLY ONCE !!!
!pip install numpy==1.26.4 --force-reinstall

In [ ]:
import cv2
import os
from deoldify.visualize import *
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')

# Define paths to preprocessed image folders (from my Google Drive)
base_path = '/content/drive/My Drive/Processed_Images/'  # Base path to your preprocessed images
high_quality_folder = os.path.join(base_path, 'High')
medium_quality_folder = os.path.join(base_path, 'Medium')
low_quality_folder = os.path.join(base_path, 'Low')

output_folder = '/content/drive/My Drive/Colorized_Images'  # Folder on Google Drive where colorized images will be saved

# Create the output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

colorizer = get_image_colorizer(artistic=True)

# Method to process, display, and colorize images
def process_and_colorize_images(folder_path, quality_category, output_folder):
    #Get the list of images in the folder
    image_files = os.listdir(folder_path)

    # Loop through each image in the folder
    for image_file in image_files:
        image_path = os.path.join(folder_path, image_file)

        #Open image using PIL and convert to RGB
        img = Image.open(image_path)
        img = img.convert('RGB')  # Ensure it's in RGB mode

        #Save the image to the same path for consistent format
        img.save(image_path)

        #Feed the preprocessed image into the DeOldify model for colorization
        colorizer.plot_transformed_image(path=image_path, render_factor=35, compare=True, watermarked=True)

        #Save the colorized image (if needed for further use)
        output_image_path = os.path.join(output_folder, f"colorized_{image_file}")
        img.save(output_image_path)  # Save the image as RGB
        print(f"Processed and saved colorized image: {output_image_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
process_and_colorize_images(high_quality_folder, 'High', output_folder)
process_and_colorize_images(medium_quality_folder, 'Medium', output_folder)
process_and_colorize_images(low_quality_folder, 'Low', output_folder)

In [ ]:
import os
from deoldify.visualize import *
from PIL import Image
from google.colab import drive
import numpy as np

# Mount Google Drive to access preprocessed images
drive.mount('/content/drive')

# Define paths to preprocessed image folders (from your Google Drive)
base_path = '/content/drive/My Drive/Processed_Images/'  # Base path to your preprocessed images
high_quality_folder = os.path.join(base_path, 'High')
medium_quality_folder = os.path.join(base_path, 'Medium')
low_quality_folder = os.path.join(base_path, 'Low')

output_folder = '/content/drive/My Drive/Colorized_Images'  # Folder on Google Drive where colorized images will be saved

# Create the output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Initialize the DeOldify colorizer model (assuming you want the artistic model)
colorizer = get_image_colorizer(artistic=True)

# Method to colorize images and save them to Google Drive
def process_and_save_colorized_images_z(folder_path, output_folder):
    # Get the list of images in the folder
    image_files = os.listdir(folder_path)

    # Loop through each image in the folder
    for image_file in image_files:
        image_path = os.path.join(folder_path, image_file)

        # Open image using PIL and convert to RGB
        img = Image.open(image_path)
        img = img.convert('RGB')  # Ensure it's in RGB mode

        # Use `plot_transformed_image()` to colorize and display the image
        colorizer.plot_transformed_image(path=image_path, render_factor=35, compare=False, watermarked=True)

        # Save the figure after it has been colorized and displayed
        fig = plt.gcf()  # Get the current figure
        output_image_path = os.path.join(output_folder, f"colorized_{image_file}")

        # Save the colorized image (the plot) to the output folder
        fig.savefig(output_image_path)  # Save the current figure as an image
        plt.close(fig)  # Close the figure to free memory

        print(f"Processed and saved colorized image: {output_image_path}")


process_and_save_colorized_images_z(high_quality_folder, output_folder)
process_and_save_colorized_images_z(medium_quality_folder, output_folder)
process_and_save_colorized_images_z(low_quality_folder, output_folder)

In [ ]:
import os
from deoldify.visualize import *
from PIL import Image
from google.colab import drive
import matplotlib.pyplot as plt

# Mount Google Drive to access preprocessed images
drive.mount('/content/drive')

# Define paths to preprocessed image folders (from your Google Drive)
base_path = '/content/drive/My Drive/Processed_Images/'  # Base path to your preprocessed images
high_quality_folder = os.path.join(base_path, 'High')
medium_quality_folder = os.path.join(base_path, 'Medium')
low_quality_folder = os.path.join(base_path, 'Low')

output_folder = '/content/drive/My Drive/Colorized2_Images'  # Folder on Google Drive where colorized images will be saved

# Create the output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Initialize the DeOldify colorizer model (assuming you want the artistic model)
colorizer = get_image_colorizer(artistic=True)

# Method to colorize images and save them to Google Drive
def process_and_save_colorized_images_x(folder_path, output_folder):
    # Get the list of images in the folder
    image_files = os.listdir(folder_path)

    # Loop through each image in the folder
    for image_file in image_files:
        image_path = os.path.join(folder_path, image_file)

        # Open image using PIL and convert to RGB
        img = Image.open(image_path)
        img = img.convert('RGB')  # Ensure it's in RGB mode

        # Use `plot_transformed_image()` to colorize and display the image
        colorizer.plot_transformed_image(path=image_path, render_factor=35, compare=False, watermarked=True)

        # Save the figure after it has been colorized and displayed
        fig = plt.gcf()  # Get the current figure

        # Set figure size to match the image dimensions to remove borders
        img_width, img_height = img.size
        fig.set_size_inches(img_width / 100.0, img_height / 100.0)  # Set the figure size (scale to inches)

        # Remove axes (no ticks or labels)
        plt.axis('off')

        output_image_path = os.path.join(output_folder, f"colorized_{image_file}")

        # Save the colorized image (the plot) to the output folder without extra borders
        fig.savefig(output_image_path, dpi=300, bbox_inches='tight', pad_inches=0)
        plt.close(fig)  # Close the figure to free memory

        print(f"Processed and saved colorized image: {output_image_path}")


process_and_save_colorized_images_x(high_quality_folder, output_folder)
process_and_save_colorized_images_x(medium_quality_folder, output_folder)
process_and_save_colorized_images_x(low_quality_folder, output_folder)


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')

# Paths to images stored in Google Drive
original_base_path = '/content/drive/My Drive/Old_Images/'  # Original images folder
processed_base_path = '/content/drive/My Drive/Processed_Images/'  # Processed images folder
colorized_base_path = '/content/drive/My Drive/Colorized2_Images/'  # Colorized images folder

# Function to get the paths of the images from the respective folders
def get_image_paths(folder_path, quality_category, image_file):

    colorized_image_file = f"colorized_{image_file}"

    # Construct full paths to each image
    original_image_path = os.path.join(original_base_path, quality_category, image_file)
    preprocessed_image_path = os.path.join(processed_base_path, quality_category, image_file)
    colorized_image_path = os.path.join(colorized_base_path, colorized_image_file)

    return original_image_path, preprocessed_image_path, colorized_image_path

# Method to display the original, preprocessed, and DeOldified images side by side
def display_comparison(original_path, preprocessed_path, colorized_path, title="Image Comparison"):
    # Open the images using PIL and convert to RGB
    original = Image.open(original_path).convert('RGB')
    preprocessed = Image.open(preprocessed_path).convert('RGB')
    colorized = Image.open(colorized_path).convert('RGB')

    # Convert the images to numpy arrays for plotting
    original_np = np.array(original)
    preprocessed_np = np.array(preprocessed)
    colorized_np = np.array(colorized)

    # Set up the plot to display the three images side by side
    plt.figure(figsize=(18, 6))

    # Display the original image
    plt.subplot(1, 3, 1)
    plt.imshow(original_np)
    plt.title("Original Image")
    plt.axis('off')

    # Display the preprocessed image
    plt.subplot(1, 3, 2)
    plt.imshow(preprocessed_np)
    plt.title("Preprocessed Image")
    plt.axis('off')

    # Display the colorized image
    plt.subplot(1, 3, 3)
    plt.imshow(colorized_np)
    plt.title("DeOldified Image")
    plt.axis('off')

    # Adjust layout and display the images
    plt.tight_layout()
    plt.show()

# Example of iterating through all images in a folder and displaying them
def process_and_display_all_images(quality_category):
    # List all image files in the "High", "Medium", "Low" subfolder
    image_files = os.listdir(os.path.join(original_base_path, quality_category))

    # Loop through each image file and display the images side by side
    for image_file in image_files:
        original_path, preprocessed_path, colorized_path = get_image_paths(
            original_base_path, quality_category, image_file)

        display_comparison(original_path, preprocessed_path, colorized_path, title=f"Comparison for {image_file}")


process_and_display_all_images('High')
process_and_display_all_images('Medium')
process_and_display_all_images('Low')

In [ ]:
import os
import numpy as np
from PIL import Image
import cv2


def ensure_directory_exists(output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Created directory: {output_folder}")

#VERY Light denoising to remove minor noise without affecting image !!!!
def apply_gaussian_blur(image):
    # Apply Gaussian Blur without over-smoothing using a small kernel size (5x5)
    blurred_image = cv2.GaussianBlur(image, (5, 5), 0)
    return blurred_image

# Post-process the image with light techniques
def gaussian_post_process_image(image):
    blurred_image = apply_gaussian_blur(image)
    return blurred_image


#Process and save post-processed images
def process_post_processing_and_save(image_path, output_folder):
    image = Image.open(image_path)
    image = image.convert('RGB')  # Ensure RGB mode
    image_np = np.array(image)

    #post-processing
    processed_image = gaussian_post_process_image(image_np)

    #Convert back to PIL for saving
    processed_image_pil = Image.fromarray(processed_image)

    #Ensure the output directory exists
    ensure_directory_exists(output_folder)

    #Save the processed image
    output_image_path = os.path.join(output_folder, f"processed_{os.path.basename(image_path)}")
    processed_image_pil.save(output_image_path)

    print(f"Processed and saved post-processed image: {output_image_path}")


def process_all_colorized_images_for_post_processing(colorized_folder, output_folder):
    image_files = os.listdir(colorized_folder)

    for image_file in image_files:
        image_path = os.path.join(colorized_folder, image_file)
        process_post_processing_and_save(image_path, output_folder)


colorized_base_path = '/content/drive/My Drive/Colorized2_Images/'
output_folder = '/content/drive/My Drive/PostProcessed_Images/'

process_all_colorized_images_for_post_processing(colorized_base_path, output_folder)


In [ ]:
import matplotlib.pyplot as plt
import cv2
from PIL import Image

base_path_old = '/content/drive/My Drive/Old_Images/'
base_path_processed = '/content/drive/My Drive/Processed_Images/'
base_path_colorized = '/content/drive/My Drive/Colorized2_Images/'
base_path_postprocessed = '/content/drive/My Drive/PostProcessed_Images/'


def display_comparison(original, preprocessed, deoldified, postprocessed, title="Image Comparison"):
    """
    Display the Original, Preprocessed, DeOldified, and Post-Processed images side by side.

    Parameters:
    - original: Original image (before any processing)
    - preprocessed: Preprocessed image (before colorization)
    - deoldified: Colorized (DeOldified) image
    - postprocessed: Post-processed image after Gaussian Blur
    """
    plt.figure(figsize=(20, 6))

    # Original image
    plt.subplot(1, 4, 1)
    plt.imshow(original)
    plt.title("Original Image")
    plt.axis('off')

    # Preprocessed image
    plt.subplot(1, 4, 2)
    plt.imshow(preprocessed)
    plt.title("Preprocessed Image")
    plt.axis('off')

    # DeOldified (colorized) image
    plt.subplot(1, 4, 3)
    plt.imshow(deoldified)
    plt.title("DeOldified (Colorized) Image")
    plt.axis('off')

    # Post-processed image (after applying Gaussian Blur)
    plt.subplot(1, 4, 4)
    plt.imshow(postprocessed)
    plt.title("Post-Processed Image")
    plt.axis('off')

    plt.tight_layout()
    plt.show()


def display_all_images_for_comparison(base_path_old, base_path_processed, base_path_colorized, base_path_postprocessed, quality_category):
    #Paths
    original_path = os.path.join(base_path_old, quality_category)
    preprocessed_path = os.path.join(base_path_processed, quality_category)
    colorized_path = base_path_colorized
    postprocessed_path = base_path_postprocessed

    # Get the images for the given quality category
    image_files = os.listdir(original_path)

    for image_file in image_files:
        original_image_path = os.path.join(original_path, image_file)
        preprocessed_image_path = os.path.join(preprocessed_path, image_file)
        colorized_image_path = os.path.join(colorized_path, f"colorized_{image_file}")
        postprocessed_image_path = os.path.join(postprocessed_path, f"processed_colorized_{image_file}")

        #Ensuring all images exist
        if not os.path.exists(original_image_path):
            print(f"Original image {image_file} not found!")
            continue
        if not os.path.exists(preprocessed_image_path):
            print(f"Preprocessed image {image_file} not found!")
            continue
        if not os.path.exists(colorized_image_path):
            print(f"Colorized image {image_file} not found!")
            continue
        if not os.path.exists(postprocessed_image_path):
            print(f"Post-processed image {image_file} not found!")
            continue

        #Load the images
        original = Image.open(original_image_path).convert('RGB')
        preprocessed = Image.open(preprocessed_image_path).convert('RGB')
        deoldified = Image.open(colorized_image_path).convert('RGB')
        postprocessed = Image.open(postprocessed_image_path).convert('RGB')

        # Display the comparison
        display_comparison(original, preprocessed, deoldified, postprocessed)



In [ ]:
display_all_images_for_comparison(base_path_old, base_path_processed, base_path_colorized, base_path_postprocessed, 'High')
display_all_images_for_comparison(base_path_old, base_path_processed, base_path_colorized, base_path_postprocessed, 'Medium')
display_all_images_for_comparison(base_path_old, base_path_processed, base_path_colorized, base_path_postprocessed, 'Low')

In [ ]:
# ----- Peak Signal-to-Noise Ratio (PSNR)

# Used to measure the quality of the image after reconstruction or colorization
# A higher PSNR means a better quality image :)


import cv2
import numpy as np

def calculate_psnr(original, colorized):
    # Convert images to grayscale (for PSNR calculation)
    original_gray = cv2.cvtColor(np.array(original), cv2.COLOR_RGB2GRAY)
    colorized_gray = cv2.cvtColor(np.array(colorized), cv2.COLOR_RGB2GRAY)

    # Calculate Mean Squared Error
    mse = np.mean((original_gray - colorized_gray) ** 2)
    if mse == 0:
        return 100  # PSNR is infinite if the images are identical

    max_pixel = 255.0
    psnr = 20 * np.log10(max_pixel / np.sqrt(mse))
    return psnr

In [ ]:
# ----- Structural Similarity Index (SSIM)

# Used to compare the structural similarity between two images and considers luminance, contrast, and texture information


from skimage.metrics import structural_similarity as ssim

def calculate_ssim(original, colorized):
    original = np.array(original)
    colorized = np.array(colorized)

    # Convert to grayscale
    original_gray = cv2.cvtColor(original, cv2.COLOR_RGB2GRAY)
    colorized_gray = cv2.cvtColor(colorized, cv2.COLOR_RGB2GRAY)

    # Compute SSIM
    score, _ = ssim(original_gray, colorized_gray, full=True)
    return score

In [ ]:
!pip install lpips

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#!ls /content/drive/My\ Drive/
#verified the existence :)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
